# 03 - Panel Orientation Simulation

Compare energy production across panel orientations and tilt angles
using pvlib-based transposition models.

In [ ]:
from solar_intelligence.data_loader import generate_synthetic_solar_data
from solar_intelligence.orientation_simulator import OrientationSimulator, RooftopScorer
from solar_intelligence.visualization import SolarVisualizer
import holoviews as hv
hv.extension('bokeh')

In [ ]:
ds = generate_synthetic_solar_data(lat=28.6, lon=77.2, start_year=2023, end_year=2023)
ghi = ds['ALLSKY_SFC_SW_DWN'].values
viz = SolarVisualizer(width=700, height=400)

## Simulate All Orientations

In [ ]:
sim = OrientationSimulator(
    latitude=28.6, longitude=77.2,
    tilt_angles=[0, 15, 30, 45],
    azimuths={'North': 0, 'East': 90, 'South': 180, 'West': 270},
)
sim_data = sim.simulate_all_orientations(ghi, year=2023)
print(f"Simulation: {len(sim_data)} rows ({len(sim_data['direction'].unique())} dirs x {len(sim_data['tilt_deg'].unique())} tilts x 12 months)")

## Optimal Orientation

In [ ]:
optimal = sim.optimal_orientation(ghi, year=2023)
for k, v in optimal.items():
    print(f"{k}: {v}")

## Direction Comparison

In [ ]:
viz.orientation_comparison_bar(sim_data, tilt=30)

## Tilt Sensitivity

In [ ]:
sensitivity = sim.tilt_sensitivity_analysis(ghi, year=2023, tilt_range=[0, 10, 20, 30, 40, 50, 60])
viz.tilt_energy_curve(sensitivity)

## Orientation Heatmap

In [ ]:
viz.orientation_heatmap(sim_data)

## Polar Plot

In [ ]:
viz.orientation_polar_plot(sim_data, tilt=30)

## Tracking Simulation

In [ ]:
single = sim.simulate_tracking(ghi, mode='single_axis')
dual = sim.simulate_tracking(ghi, mode='dual_axis')
print(f"Fixed optimal: {optimal['annual_energy_kwh']:.0f} kWh")
print(f"Single-axis:   {single['annual_energy_kwh']:.0f} kWh (+{single['gain_vs_fixed_pct']:.1f}%)")
print(f"Dual-axis:     {dual['annual_energy_kwh']:.0f} kWh (+{dual['gain_vs_fixed_pct']:.1f}%)")

## Rooftop Suitability Score

In [ ]:
scorer = RooftopScorer(latitude=28.6, longitude=77.2)
avg_ghi = float(ds['ALLSKY_SFC_SW_DWN'].mean())
result = scorer.score(
    avg_daily_ghi=avg_ghi,
    optimal_tilt=optimal['best_tilt'],
    roof_tilt=25,
    variability_index=0.12,
    avg_temperature=25,
)
print(f"Score: {result['total_score']}/100 ({result['rating']})")
for comp, val in result['components'].items():
    print(f"  {comp}: {val}")
if result['recommendations']:
    print(f"Recommendations: {result['recommendations']}")